## Task 3: NLP analysis, RAG pipeline, and agentic automation

### Preprocess log data using NLP techniques such as TF-IDF or word embeddings

In [41]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

logs = pd.read_csv('../data/application_logs.csv', parse_dates=['timestamp'])

# Remove INFO logs because they don't help separate between business and infrastructure
issues = logs[~logs.log_level.isin(["INFO"])].copy()

# Convert the log messages into TF-IDF values
log_vectorizer = TfidfVectorizer(stop_words='english', max_features=100)
log_vectors = log_vectorizer.fit_transform(issues['log_message'])

print(f'TF-IDF shape: {log_vectors.shape}')
display(issues.head())

TF-IDF shape: (3761, 100)


,timestamp,log_level,log_message
1,2025-10-20 00:01:00,ERROR,ERROR: NEFT transaction failed. Transaction ID...
2,2025-10-20 00:01:00,ERROR,ERROR: Database connection pool exhausted. Con...
3,2025-10-20 00:03:00,ERROR,ERROR: IMPS transaction failed. Transaction ID...
4,2025-10-20 00:03:00,WARNING,WARNING: Memory usage approaching threshold on...
5,2025-10-20 00:04:00,ERROR,ERROR: High CPU utilization detected on server...


### Apply unsupervised clustering to group logs into infrastructure issues and business KPI issues

In [ ]:
import pandas as pd
from sklearn.cluster import KMeans

logs = pd.read_csv('../data/application_logs.csv')

kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)

issues['cluster'] = kmeans.fit_predict(log_vectors)

# Assign the cluster that mentions 'srv' most to infrastructure, and the other one to business
srv_share = issues.groupby('cluster').log_message.apply(lambda s: s.str.contains('srv').mean())
infra = srv_share.idxmax()
issues['issue_group'] = issues.cluster.map({infra: 'Infrastructure issue', 1 - infra: 'Business KPI issue'})

# Get the most important words from each cluster
terms = log_vectorizer.get_feature_names_out()
for c in range(2):
    top = terms[kmeans.cluster_centers_[c].argsort()[-10:][::-1]]

    # Print each cluster's name, size, and top words
    print(f'{issues.issue_group[issues.cluster == c].iloc[0]}: {(issues.cluster == c).sum()} logs')
    print('Top words: ', ', '.join(top))

# Check which cluster each message type landed in
display(pd.crosstab(issues.log_message.str.replace(r'\d+', 'N', regex=True), issues.issue_group))

Business KPI issue: 1736 logs
Top words:  transaction, id, failed, reason, error, time, gateway, duplicate, warning, authentication
Infrastructure issue: 2025 logs
Top words:  srv, error, api, current, detected, warning, disk, bottleneck, database, pool


issue_group,Business KPI issue,Infrastructure issue
log_message,,
CRITICAL: Network timeout on srv-N. Latency: Nms,0,294
ERROR: API rate limit exceeded for endpoint /api/transactions,0,286
ERROR: Customer authentication failed. User ID: USRN,286,0
ERROR: Database connection pool exhausted. Connections: N,0,283
ERROR: Disk I/O bottleneck detected on srv-N,0,284
ERROR: High CPU utilization detected on server srv-N. Current: N%,0,304
ERROR: IMPS transaction failed. Transaction ID: TXNN. Reason: Invalid account,264,0
ERROR: NEFT transaction failed. Transaction ID: TXNN. Reason: Gateway timeout,292,0
ERROR: RTGS transaction failed. Transaction ID: TXNN. Reason: Insufficient funds,279,0


This table shows that each log message type was put into one of the two clusters. The infrastructure cluster has log messages involving infrastructure problems such as network timeouts, rate limits, and database problems. The business cluster shows log messages for events that directly impacted customers.

### Build a knowledge base from historical incidents and documentation

In [33]:
import re
import pandas as pd

incidents = pd.read_csv('../data/incident_reports.csv', parse_dates=['timestamp'])

with open('../knowledge_base/incident_knowledge_base.txt') as file:
    runbook_text = file.read()

knowledge_documents = []

# Parse each entry and add as a document
entries = runbook_text.split('=' * 80)
for entry in entries:
    entry = entry.strip()
    if not entry:
        continue
    title_match = re.search(r'TITLE:\s*(.+)', entry)
    title = title_match.group(1).strip()
    knowledge_documents.append({
        'title': title,
        'source': 'incident_knowledge_base.txt',
        'text': entry,
    })

# Add each non-normal incident from incident_reports.csv as a document
non_normal_incidents = incidents[incidents['incident_type'] != 'Normal']
for i, incident in non_normal_incidents.iterrows():
    text = (
        f"Historical incident {incident['incident_id']} at {incident['timestamp']}. "
        f"Type: {incident['incident_type']}. Severity: {incident['severity']}. "
        f"CPU: {incident['cpu_utilization']:.1f}%. "
        f"Memory: {incident['memory_usage']:.1f}%. "
        f"Database pool: {incident['db_connection_pool_usage']:.1f}%. "
        f"NEFT success rate: {incident['neft_success_rate']:.1f}%. "
        f"Application error rate: {incident['application_error_rate']:.1f}."
    )

    knowledge_documents.append({
        'title': incident['incident_id'],
        'source': 'incident_reports.csv',
        'text': text,
    })

knowledge_base = pd.DataFrame(knowledge_documents)
print(f'Number of documents: {len(knowledge_base)}')
knowledge_base.head(8)

Number of documents: 34


,title,source,text
0,High CPU Utilization - Root Cause Analysis,incident_knowledge_base.txt,TITLE: High CPU Utilization - Root Cause Analy...
1,Database Connection Pool Exhaustion,incident_knowledge_base.txt,TITLE: Database Connection Pool Exhaustion\nCO...
2,NEFT Transaction Failures,incident_knowledge_base.txt,TITLE: NEFT Transaction Failures\nCONTENT:\nNE...
3,Memory Usage Spike,incident_knowledge_base.txt,TITLE: Memory Usage Spike\nCONTENT:\nSudden me...
4,API Rate Limit Exceeded,incident_knowledge_base.txt,TITLE: API Rate Limit Exceeded\nCONTENT:\nAPI ...
5,INC000012,incident_reports.csv,Historical incident INC000012 at 2025-10-20 11...
6,INC000014,incident_reports.csv,Historical incident INC000014 at 2025-10-20 13...
7,INC000015,incident_reports.csv,Historical incident INC000015 at 2025-10-20 14...


This separates all the entries in `incident_knowledge_base.txt`, extracts their title and content, and adds them to a dataframe. Then it does the same thing with non-normal incidents from `incident_reports.csv`. This is ready for processing and turning it into a RAG retrieval system.

### Integrate a retrieval-augmented generation (RAG) pipeline to fetch relevant incidents and documents

In [42]:
from sklearn.metrics.pairwise import cosine_similarity

# Create TF-IDF vectors for the knowledge base
knowledge_vectorizer = TfidfVectorizer(stop_words='english')
knowledge_vectors = knowledge_vectorizer.fit_transform(knowledge_base['text'])

# Find the most similar documents using cosine similarity and TF-IDF vectors
def retrieve_documents(question, top_k=3):
    question_vector = knowledge_vectorizer.transform([question])
    scores = cosine_similarity(question_vector, knowledge_vectors).flatten()
    top_indexes = scores.argsort()[-top_k:][::-1]
    results = knowledge_base.iloc[top_indexes].copy()
    results['similarity'] = scores[top_indexes]
    return results

retrieved = retrieve_documents('What causes NEFT transaction failures?')
display(retrieved[['title', 'similarity']].round(2))

,title,similarity
2,NEFT Transaction Failures,0.38
0,High CPU Utilization - Root Cause Analysis,0.13
1,Database Connection Pool Exhaustion,0.05


The `retrieve_documents` function is able to use cosine similarity to find the knowledge base vectors closest to the input question's vector and then return the most similar documents. It is ready for an assistant to use.

### Implement an AI assistant using LangGraph or LangChain to answer queries

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv('../environment/.env')

True

In [38]:
from textwrap import dedent
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_template(
    dedent(
        """
        You are a banking operations assistant. Answer using only the context below.
        Context: {context}
        Question: {question}
        Give the likely root cause and 2-3 recommended actions.
        """
    ).strip()
)

llm = ChatOpenAI(model='gpt-4o-mini', base_url= 'https://openai.vocareum.com/v1', temperature=0)
chain = prompt | llm | StrOutputParser()

def ask(question, top_k=3):
    docs = retrieve_documents(question, top_k=top_k)
    context = '\n\n'.join(f'[{r.title}]\n{r.text}' for r in docs.itertuples())
    return chain.invoke({'context': context, 'question': question})

print(ask('Our CPU is at 92% and the connection pool is exhausted. What should I do first?'))

The likely root cause of the CPU being at 92% and the connection pool being exhausted is likely due to insufficient pool size for the current load or connection leaks in application code.

Recommended actions:
1. Increase the connection pool size temporarily to accommodate the current load.
2. Review application logs for connection leak patterns to identify and fix any issues.
3. Optimize any slow-running queries that may be contributing to the high CPU usage and connection pool exhaustion.


This simple AI assistant is able to use the `retrieve_documents` function to get the relevant documents on the problem and give solutions.

### Generate AI-based root cause summaries and recommendations

In [40]:
# Turn a real incident_reports row into a question, then pass it to the assistant
def summarize_incident(incident):
    question = dedent(
        f"""
        Incident {incident['incident_id']} ({incident['incident_type']},
        severity {incident['severity']}). CPU {incident['cpu_utilization']:.1f}%,
        memory {incident['memory_usage']:.1f}%, DB pool {incident['db_connection_pool_usage']:.1f}%,
        error rate {incident['application_error_rate']},
        NEFT success {incident['neft_success_rate']:.1f}%.
        What is the likely root cause and what are the most likely solutions?
        """
    ).strip()
    return ask(question)

# Test two critical incidents
critical = incidents[incidents['is_critical'] == 1]
for i, incident in critical.head(2).iterrows():
    print(f"--- {incident['incident_id']} ---")
    print(summarize_incident(incident))
    print()

--- INC000012 ---
**Likely Root Cause:** The incident INC000012 indicates a resource exhaustion issue, characterized by high CPU (87.7%), high memory usage (87.7%), and a very high database pool usage (93.5%). This suggests that the system is under heavy load, potentially due to increased transaction volume or inefficient resource management.

**Recommended Actions:**
1. **Optimize Application Performance:** Review and optimize the application code to reduce CPU and memory usage, addressing any inefficiencies that may be causing excessive resource consumption.
   
2. **Scale Resources:** Increase the available CPU and memory resources to handle the current load more effectively, ensuring that the system can accommodate peak transaction volumes without degradation in performance.

3. **Monitor and Adjust Database Pool Settings:** Analyze the database pool configuration and consider increasing the pool size or optimizing database queries to improve performance and reduce the high usage p